# Stage 4: RAG Analysis Pipeline

> **Runtime Prerequisite:** This notebook requires **Stages 2-3** to have been executed
> so that `processed_holdings.json` and the ChromaDB vector store are available.

## Pipeline Architecture
| Component | Implementation |
|---|---|
| **Generator** | Gemini 3.1 Flash-Lite Preview (via Google AI Studio) |
| **Evaluator** | GPT-4o-mini (via GitHub Models / OpenAI) |
| **Judge** | Gemini 2.5 Pro (via Google AI Studio) |
| **Retrieval** | EnsembleRetriever (BM25 + ChromaDB Vector) |
| **Reranking** | FlashrankRerank (narrow=8, wide=20) |
| **Synthesis** | Routed Stuff / Map-Reduce chain |
| **Filters** | Entity-scoped + Temporal + Fund-diversity |
| **Audit** | `system_audit_trail.json` with accession numbers per query |

In [5]:
# ── Set Working Directory ─────────────────────────────────────────────────────
# Ensure working directory is the Model/ folder so all relative paths
# (./data/, ./vector_db/, etc.) resolve consistently across notebooks.
import os, pathlib

_cwd = pathlib.Path.cwd()
# If VS Code opened the workspace root, navigate into Model/
if not (_cwd / "data").exists() and (_cwd / "Model" / "data").exists():
    os.chdir(_cwd / "Model")

os.makedirs("data", exist_ok=True)
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\user\Documents\1fintech\GenAI\Individual Assignment 2\Model


In [6]:
!pip install langchain langchain-community langchain-huggingface langchain-chroma \
             sentence-transformers chromadb rank_bm25 tabulate flashrank openai \
             google-generativeai

You should consider upgrading via the 'C:\Users\user\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [7]:
import os
import re
import json
import textwrap
import time
import pathlib
import logging
from datetime import datetime, timezone
from dataclasses import dataclass

# Suppress verbose httpx / httpcore request logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

import google.generativeai as genai
import openai as _openai
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain_core.documents import Document
from langchain.prompts import PromptTemplate

# ── Paths ──────────────────────────────────────────────────────────────────────
CHROMA_DIR      = "./vector_db/local_chroma_storage"
PROCESSED_JSON  = "./data/processed_holdings.json"
FILINGS_BASE    = "./data/13f_filings/sec-edgar-filings"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL_NAME = "gemini-3.1-flash-lite-preview"
AUDIT_LOG_PATH  = "./system_audit_trail.json"

print("Imports OK.")

C:\Users\user\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\user\AppData\Local\Temp\ipykernel_13176\1583406613.py:15: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Imports OK.


In [8]:
# ── Gemini API setup (generator: gemini-3.1-flash-lite-preview) ────────────────
_GEMINI_KEY_FILE = pathlib.Path(__file__).parent / "gemini_api_key.txt" if "__file__" in dir() \
                   else pathlib.Path("gemini_api_key.txt")

GEMINI_API_KEY = ""
if _GEMINI_KEY_FILE.exists():
    _raw_g = _GEMINI_KEY_FILE.read_text(encoding="utf-8").strip()
    if _raw_g and _raw_g != "PASTE_YOUR_GEMINI_API_KEY_HERE":
        GEMINI_API_KEY = _raw_g

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "Gemini API key not found. Either:\n"
        "  1. Paste your key into  Model/gemini_api_key.txt  (one line, no quotes), or\n"
        "  2. Set the environment variable: $env:GEMINI_API_KEY = 'your_key'"
    )

genai.configure(api_key=GEMINI_API_KEY)

_gemini_generator = genai.GenerativeModel(
    model_name=GENERATOR_MODEL_NAME,
    generation_config=genai.GenerationConfig(
        temperature=0.0,
        max_output_tokens=1024,
    ),
)
print(f"Gemini API key loaded. Generator: {GENERATOR_MODEL_NAME}")

JUDGE_MODEL_NAME = "gemini-2.5-pro"
JUDGE_MODEL      = JUDGE_MODEL_NAME
_gemini_judge = genai.GenerativeModel(
    model_name=JUDGE_MODEL_NAME,
    generation_config=genai.GenerationConfig(
        temperature=0.0,
        max_output_tokens=65536,
    ),
)

# ── GitHub Models endpoint (evaluator: GPT-4o-mini) ──────────────────────────
_GITHUB_TOKEN_FILE = pathlib.Path(__file__).parent / "github_token.txt" if "__file__" in dir() \
                     else pathlib.Path("github_token.txt")

GITHUB_TOKEN = ""
if _GITHUB_TOKEN_FILE.exists():
    _raw_gh = _GITHUB_TOKEN_FILE.read_text(encoding="utf-8").strip()
    if _raw_gh and _raw_gh != "PASTE_YOUR_GITHUB_TOKEN_HERE":
        GITHUB_TOKEN = _raw_gh

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

if not GITHUB_TOKEN:
    raise EnvironmentError(
        "GitHub token not found. Either:\n"
        "  1. Paste your token into  Model/github_token.txt  (one line, no quotes), or\n"
        "  2. Set the environment variable: $env:GITHUB_TOKEN = 'your_token'"
    )

_github_client = _openai.OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=GITHUB_TOKEN,
)

EVALUATOR_MODEL = "gpt-4o-mini"

# ── Rate-limit tracking ──────────────────────────────────────────────────────
_gemini_gen_call_count = 0
_evaluator_call_count = 0
_judge_call_count = 0
_GEMINI_GEN_INTER_CALL_DELAY = 2
_EVALUATOR_INTER_CALL_DELAY = 2
_JUDGE_INTER_CALL_DELAY = 2

# ── LLM-as-Judge prompt template ─────────────────────────────────────────────
LLM_JUDGE_TEMPLATE = """You are an expert financial AI evaluator assessing RAG pipeline quality.

Rate the ANSWER below on a 1-5 scale for ACCURACY, SPECIFICITY, and GROUNDING.
Use the PROVIDED CONTEXT as the ground-truth reference for 13F filing data.

CONTEXT (retrieved 13F filing chunks — treat as ground truth):
{context}

QUESTION:
{question}

ANSWER TO EVALUATE:
{answer}

SCORING RUBRIC:
5 = Excellent: answer cites specific data (fund names, share counts, CUSIPs, dates) that are
    directly confirmed by the context. Claims are precise and verifiable.
    Minor formatting differences do not reduce the score.
4 = Good: answer is mostly correct with specific citations. May have one minor inaccuracy
    or omission, but core factual claims are supported by the context.
3 = Adequate: answer is directionally correct and addresses the question, but is missing
    some key specifics that ARE available in the context, or includes partial data.
2 = Poor: answer makes specific claims that contradict the context, OR gives a generic
    response when the context contains specific data that should have been cited.
1 = Fail: answer is entirely wrong, fabricates data not in context, gives an error message,
    OR refuses to answer when the context clearly contains the relevant information.

IMPORTANT for scoring:
- Credit partial answers: if 3 of 4 data points are correct, score 4 not 2.
- A local model may phrase things differently than a cloud model — judge the factual
  content, not the prose style.
- PARTIAL CONTEXT: The context may contain data from DIFFERENT funds or periods than the
  question asks about. If the question asks about Fund X but the context only has Fund Y
  data, an answer that (a) states Fund X data is unavailable AND (b) analyses the Fund Y
  data that IS present should score 3-4. A blanket refusal like 'Insufficient data' with
  no analysis of available data should score 2.
- Fabrication check: an answer that invents specific numbers, dates, or fund names not
  present anywhere in the context should score 1, even if the claims sound plausible.
- An answer that correctly states 'data not available' when the context truly contains
  NO relevant filing data at all should score 4-5 (appropriate refusal).
- Generic/vague answers score lower than specific, data-backed answers.

Respond with ONLY a valid JSON object — no extra text before or after:
{{"score": <integer 1-5>, "rationale": "<one concise sentence>"}}"""

LLM_JUDGE_PROMPT = PromptTemplate(
    template=LLM_JUDGE_TEMPLATE,
    input_variables=["context", "question", "answer"],
)


def _build_judge_prompt(response, question, context_docs):
    """Build the judge prompt text (shared by both evaluator and judge)."""
    context_text = "\n\n---\n\n".join(
        d.page_content[:1500] for d in context_docs
    ) if context_docs else "(no context retrieved — baseline evaluation)"
    return LLM_JUDGE_PROMPT.format(
        context=context_text,
        question=question,
        answer=response[:1500],
    )


def _call_github_model(model_name, prompt_text, role_label, call_delay):
    """Call a GitHub Models endpoint and parse JSON score response."""
    _MAX_RETRIES = 3
    for _attempt in range(1, _MAX_RETRIES + 1):
        try:
            resp = _github_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": "You are an expert financial AI evaluator. Respond with ONLY valid JSON."},
                    {"role": "user", "content": prompt_text},
                ],
                max_tokens=256,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            json_match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
                score = max(1, min(5, int(parsed.get("score", 3))))
                rationale = str(parsed.get("rationale", "No rationale provided."))
                return score, rationale
            raise ValueError(f"No JSON found in {model_name} response: {raw[:300]}")
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "rate" in err_str.lower() or "quota" in err_str.lower()
            if is_rate_limit and _attempt < _MAX_RETRIES:
                wait = call_delay * _attempt * 3
                print(f"\n  [RATE LIMIT] {role_label} ({model_name}) attempt {_attempt}/{_MAX_RETRIES} "
                      f"hit rate limit. Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            raise


def _call_gemini_judge(prompt_text, role_label, call_delay):
    """Call Gemini 2.5 Pro as judge via Google AI Studio and parse JSON score response."""
    _MAX_RETRIES = 3
    for _attempt in range(1, _MAX_RETRIES + 1):
        try:
            system = "You are an expert financial AI evaluator. Respond with ONLY valid JSON."
            full_prompt = f"{system}\n\n{prompt_text}"

            gen_config = genai.GenerationConfig(
                temperature=0.0,
                max_output_tokens=65536,
            )
            resp = _gemini_judge.generate_content(
                full_prompt,
                generation_config=gen_config,
            )
            raw = None
            try:
                raw = resp.text.strip()
            except (ValueError, AttributeError):
                if resp.candidates and resp.candidates[0].content.parts:
                    raw = resp.candidates[0].content.parts[0].text.strip()
            if not raw:
                _reason = getattr(resp.candidates[0], 'finish_reason', 'unknown') if resp.candidates else 'no_candidates'
                raise ValueError(f"Gemini returned empty response (finish_reason={_reason})")
            json_match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
                score = max(1, min(5, int(parsed.get("score", 3))))
                rationale = str(parsed.get("rationale", "No rationale provided."))
                return score, rationale
            raise ValueError(f"No JSON found in {JUDGE_MODEL} response: {raw[:300]}")
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "RESOURCE_EXHAUSTED" in err_str
            is_empty_response = "empty response" in err_str or "finish_reason" in err_str
            if (is_rate_limit or is_empty_response) and _attempt < _MAX_RETRIES:
                wait = call_delay * _attempt * 3
                _label = "RATE LIMIT" if is_rate_limit else "EMPTY RESPONSE"
                print(f"\n  [{_label}] {role_label} ({JUDGE_MODEL}) attempt {_attempt}/{_MAX_RETRIES} "
                      f"— {err_str[:120]}. Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            raise


def _score_faithfulness_keywords(response, expect_no_data, required_terms):
    """Keyword-count fallback scorer (used only when both judges fail)."""
    resp_lower = response.lower()
    if expect_no_data:
        refusal_signals = [
            "not available", "no information", "does not exist",
            "not found", "not present", "cannot find", "no position",
            "not in", "not mentioned", "no data",
        ]
        return 5 if any(s in resp_lower for s in refusal_signals) else 1
    if not required_terms:
        return 3
    hits = [t for t in required_terms if t.lower() in resp_lower]
    ratio = len(hits) / len(required_terms)
    if ratio == 1.0:   return 5
    if ratio >= 0.75:  return 4
    if ratio >= 0.5:   return 3
    if ratio > 0:      return 2
    return 1


def score_evaluator(response, question, context_docs, expect_no_data=False, required_terms=None):
    """GPT-4o-mini evaluator via GitHub Models — faithfulness scorer."""
    global _evaluator_call_count

    prompt_text = _build_judge_prompt(response, question, context_docs)

    if _evaluator_call_count > 0:
        time.sleep(_EVALUATOR_INTER_CALL_DELAY)

    try:
        score, rationale = _call_github_model(
            EVALUATOR_MODEL, prompt_text, "Evaluator", _EVALUATOR_INTER_CALL_DELAY
        )
        _evaluator_call_count += 1
        return score, rationale, "gpt4omini_evaluator"
    except Exception as exc:
        print(f"  [WARN] Evaluator ({EVALUATOR_MODEL}) failed: {str(exc)[:150]}")
        fb_score = _score_faithfulness_keywords(response, expect_no_data, required_terms or [])
        return fb_score, f"[FALLBACK-KEYWORD] Evaluator error: {exc}", "keyword_fallback"


def score_judge(response, question, context_docs, expect_no_data=False, required_terms=None):
    """Gemini 2.5 Pro judge via Google AI Studio — faithfulness scorer."""
    global _judge_call_count

    prompt_text = _build_judge_prompt(response, question, context_docs)

    if _judge_call_count > 0:
        time.sleep(_JUDGE_INTER_CALL_DELAY)

    try:
        score, rationale = _call_gemini_judge(
            prompt_text, "Judge", _JUDGE_INTER_CALL_DELAY
        )
        _judge_call_count += 1
        return score, rationale, "gemini2_judge"
    except Exception as exc:
        print(f"  [WARN] Judge ({JUDGE_MODEL}) failed: {str(exc)[:150]}")
        fb_score = _score_faithfulness_keywords(response, expect_no_data, required_terms or [])
        return fb_score, f"[FALLBACK-KEYWORD] Judge error: {exc}", "keyword_fallback"


def score_dual(response, question, context_docs, expect_no_data=False, required_terms=None):
    """Run both evaluator and judge, return averaged score."""
    e_score, e_rat, e_method = score_evaluator(response, question, context_docs, expect_no_data, required_terms)
    j_score, j_rat, j_method = score_judge(response, question, context_docs, expect_no_data, required_terms)
    avg = round((e_score + j_score) / 2, 1)
    return avg, e_score, e_rat, e_method, j_score, j_rat, j_method


def grounding_check(response, filings_base):
    """Scans source full-submission.txt files for any 9-char CUSIP in the response."""
    resp_lower = response.lower()

    refusal_phrases = [
        "not present", "not available", "no information",
        "does not exist", "not found", "insufficient data",
        "no data", "cannot find",
    ]
    is_refusal = any(p in resp_lower for p in refusal_phrases)

    _all_9char = set(re.findall(r'\b[A-Z0-9]{9}\b', response.upper()))
    cusips_in_response = {t for t in _all_9char if re.search(r'\d', t)}
    if not cusips_in_response:
        if is_refusal:
            return None, "Refusal/empty answer — grounding indeterminate (no CUSIPs to verify)."
        return True, "No specific CUSIPs to verify (narrative answer)."

    corpus = ""
    for root, _, files in os.walk(filings_base):
        for fname in files:
            if fname == "full-submission.txt":
                try:
                    with open(os.path.join(root, fname), "r",
                              encoding="utf-8", errors="ignore") as f:
                        corpus += f.read(5_000_000).upper()
                except OSError:
                    continue

    verified   = [c for c in cusips_in_response if c in corpus]
    unverified = [c for c in cusips_in_response if c not in corpus]

    grounded = len(unverified) == 0
    parts = []
    if verified:   parts.append(f"Verified CUSIPs: {verified}")
    if unverified: parts.append(f"UNVERIFIED (possible hallucination): {unverified}")

    return grounded, " | ".join(parts) if parts else "No verifiable identifiers found."


def _call_gemini(system_content, user_content, max_tokens=1024):
    """Call Gemini 3.1 Flash-Lite as the RAG answer generator."""
    global _gemini_gen_call_count

    if _gemini_gen_call_count > 0:
        time.sleep(_GEMINI_GEN_INTER_CALL_DELAY)

    prompt = f"{system_content}\n\n{user_content}"

    for _attempt in range(1, 4):
        try:
            gen_config = genai.GenerationConfig(
                temperature=0.0,
                max_output_tokens=max_tokens,
            )
            resp = _gemini_generator.generate_content(
                prompt,
                generation_config=gen_config,
            )
            _gemini_gen_call_count += 1
            return resp.text.strip()
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "RESOURCE_EXHAUSTED" in err_str
            if _attempt < 3:
                wait = 5 * _attempt if not is_rate_limit else 10 * _attempt
                print(f"    [GEMINI GEN RETRY] attempt {_attempt}/3: {err_str[:100]}. "
                      f"Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            return f"[GEMINI ERROR] {exc}"

print(f"Generator  : {GENERATOR_MODEL_NAME} via Google AI Studio (Google DeepMind)")
print(f"Evaluator  : {EVALUATOR_MODEL} via GitHub Models (OpenAI)")
print(f"Judge      : {JUDGE_MODEL} via Google AI Studio (Google DeepMind)")
print(f"→ Two different providers (Google / OpenAI) — no self-grading bias.")

Gemini API key loaded. Generator: gemini-3.1-flash-lite-preview
Generator  : gemini-3.1-flash-lite-preview via Google AI Studio (Google DeepMind)
Evaluator  : gpt-4o-mini via GitHub Models (OpenAI)
Judge      : gemini-2.5-pro via Google AI Studio (Google DeepMind)
→ Two different providers (Google / OpenAI) — no self-grading bias.


In [9]:
# ── Chunk pre-filter ──────────────────────────────────────────────────────────
def _prefilter_chunks(question, docs):
    keywords = [w.lower() for w in question.split() if len(w) > 3]
    filtered = []
    for d in docs:
        lines = d.page_content.split("\n")
        header_lines = []
        data_lines = []
        for line in lines:
            stripped = line.strip()
            if stripped.startswith("|"):
                if "nameOfIssuer" in line or ":-" in line or "--" in line:
                    header_lines.append(line)
                else:
                    data_lines.append(line)
            else:
                header_lines.append(line)
        matched_rows = [r for r in data_lines if any(k in r.lower() for k in keywords)]
        if matched_rows:
            filtered.append("\n".join(header_lines + matched_rows))
        else:
            filtered.append(d.page_content)
    return filtered

# ── Chain routing: Stuff for exact-lookup, Map-Reduce for synthesis ───────────
MAP_REDUCE_TYPES = {
    "Cross-Fund Sector Synthesis",
    "Comparative / Deep Context",
    "Sector Concentration",
}

def run_stuff_chain(question, docs):
    filtered_chunks = _prefilter_chunks(question, docs)
    context = "\n\n---\n\n".join(filtered_chunks)
    system = (
        "You are a precise financial analyst. "
        "Use ONLY the 13F filing data provided below — no outside knowledge. "
        "Extract data directly from the tables and metadata headers. "
        "If the specific data point is absent from the provided excerpts, "
        "state: 'This information is not present in the retrieved filing data.' "
        "However, if the data is present for DIFFERENT funds or periods than asked about, "
        "report what IS available and note which requested entities are missing. "
        "CRITICAL: Copy ALL numerical values, share counts, CUSIPs, dates, and fund names "
        "VERBATIM from the context. Do not paraphrase, round, or restate them. "
        "You MUST cite the exact fund name as it appears in the context header (e.g., "
        "'Fund: Treasurer of the State of North Carolina') and the accession number "
        "for every claim. Never use generic labels like 'Fund 1' or 'the fund'."
    )
    user = (
        f"Filing data:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer using ONLY the data above. "
        "Cite exact fund name, accession number, share count, CUSIP, and filing date:"
    )
    return _call_gemini(system, user, max_tokens=1024)

def run_map_reduce_chain(question, docs):
    filtered_chunks = _prefilter_chunks(question, docs)
    map_system = (
        "You are a financial analyst reviewing a single 13F filing excerpt. "
        "First, extract the fund name, filing date, period of report, and accession "
        "number from the metadata header. Then extract the holdings data most relevant "
        "to the question — include issuer names, CUSIPs, share counts, market values, "
        "and any sector/industry identifiers present. Copy ALL numerical values and "
        "identifiers VERBATIM. Always extract what IS present — even if the excerpt "
        "is from a different fund than the one asked about. "
        "If the excerpt is completely empty or contains no tabular data, "
        "state: 'No holdings data in this excerpt.'"
    )
    summaries = []
    for i, chunk_text in enumerate(filtered_chunks):
        map_user = (
            f"Context (excerpt {i+1} of {len(filtered_chunks)}):\n{chunk_text}\n\n"
            f"Question: {question}\n\n"
            "Extract: fund name, accession number, filing date, period of report, "
            "and all holdings data relevant to the question above "
            "(issuer names, CUSIPs, values, share counts):"
        )
        summary = _call_gemini(map_system, map_user, max_tokens=400)
        summaries.append(summary)

    reduce_system = (
        "You are a meticulous financial analyst consolidating multiple 13F filing excerpts. "
        "Use ONLY the summaries below — no outside knowledge. "
        "Preserve all numerical values, CUSIPs, share counts, dates, and fund names "
        "EXACTLY as stated. Do not invent, round, or paraphrase figures. "
        "IMPORTANT: If the question asks about specific funds that are NOT present in "
        "the summaries, state which funds were requested but unavailable, then analyse "
        "the data that IS available. Identify trends: compare values across quarters "
        "if multiple periods are present, note which positions grew or shrank. "
        "Always cite the exact fund name and accession number for every claim."
    )
    joined = "\n\n---\n\n".join(f"Excerpt {i+1}:\n{s}" for i, s in enumerate(summaries))
    reduce_user = (
        f"Summaries from {len(filtered_chunks)} retrieved chunks:\n{joined}\n\n"
        f"Question: {question}\n\n"
        "Final consolidated answer (cite exact fund names, accession numbers, share counts, "
        "CUSIPs, and filing dates for every claim. If multiple quarters are present, "
        "compare values across periods to identify trends):"
    )

    return _call_gemini(reduce_system, reduce_user, max_tokens=1024)

print("Chain routing ready.")
print(f"  Stuff chain     : exact-lookup queries")
print(f"  Map-Reduce chain: {MAP_REDUCE_TYPES}")

Chain routing ready.
  Stuff chain     : exact-lookup queries
  Map-Reduce chain: {'Sector Concentration', 'Comparative / Deep Context', 'Cross-Fund Sector Synthesis'}


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

# ── Initialise embeddings and vector store ────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
)

# ── Rebuild BM25 + EnsembleRetriever + FlashrankRerank (dual) ─────────────────
print("Rebuilding BM25 retriever from source JSON ...")
with open(PROCESSED_JSON, "r", encoding="utf-8") as f:
    _raw_docs = json.load(f)
_documents = [
    Document(page_content=e["text"], metadata=e["metadata"])
    for e in _raw_docs
]

bm25_retriever = BM25Retriever.from_documents(_documents)
bm25_retriever.k = 10

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

try:
    compressor_narrow = FlashrankRerank(top_n=8)
    compressor_wide   = FlashrankRerank(top_n=20)
    reranking_retriever = ContextualCompressionRetriever(
        base_compressor=compressor_narrow,
        base_retriever=hybrid_retriever,
    )
    reranking_retriever_wide = ContextualCompressionRetriever(
        base_compressor=compressor_wide,
        base_retriever=hybrid_retriever,
    )
    USE_RERANKER = True
    print("FlashrankRerank       : ACTIVE (narrow=8, wide=20)")
except Exception as _exc:
    reranking_retriever = hybrid_retriever
    reranking_retriever_wide = hybrid_retriever
    USE_RERANKER = False
    print(f"[WARN] FlashrankRerank unavailable ({_exc}); using EnsembleRetriever only")

RETRIEVAL_LABEL = "Hybrid BM25+Vector + FlashrankRerank" if USE_RERANKER else "Hybrid BM25+Vector"

# ── Entity-scoped filter ──────────────────────────────────────────────────────
_FUND_ALIAS_MAP = {}
_all_fund_names = sorted(set(
    d.metadata.get("fund_name", "") for d in _documents if d.metadata.get("fund_name")
))
for _fn in _all_fund_names:
    _fn_lower = _fn.lower()
    _FUND_ALIAS_MAP[_fn_lower] = _fn
    if "(" in _fn:
        _base = _fn[:_fn.index("(")].strip()
        _FUND_ALIAS_MAP[_base.lower()] = _fn
        _paren = _fn[_fn.index("(") + 1 : _fn.index(")")].strip()
        if len(_paren) > 3:
            _FUND_ALIAS_MAP[_paren.lower()] = _fn
    _cleaned = _fn.split("(")[0] if "(" in _fn else _fn
    for _suffix in ["Inc", "LLC", "Corp", "Ltd", "PLC", "AG", "N.A.", "Co"]:
        _cleaned = _cleaned.replace(_suffix, "")
    _words = _cleaned.strip().split()
    if len(_words) >= 2:
        _two_word = " ".join(_words[:2]).lower().strip()
        if len(_two_word) > 5:
            _FUND_ALIAS_MAP[_two_word] = _fn

def _extract_fund_entities(question):
    q_lower = question.lower()
    _BROAD_SIGNALS = ["all funds", "which funds", "across all", "all available", "any funds"]
    if any(sig in q_lower for sig in _BROAD_SIGNALS):
        return set()
    matched = set()
    for alias in sorted(_FUND_ALIAS_MAP.keys(), key=len, reverse=True):
        if len(alias) <= 4:
            continue
        if alias in q_lower:
            matched.add(_FUND_ALIAS_MAP[alias])
    return matched

def _entity_filter(docs, target_funds):
    if not target_funds:
        return docs
    filtered = [d for d in docs if d.metadata.get("fund_name", "") in target_funds]
    return filtered if filtered else docs

# ── Temporal filter ───────────────────────────────────────────────────────────
def _temporal_filter(docs, question):
    q_lower = question.lower()
    PERIOD_MAP = {
        ("feb 2026", "february 2026", "q4 2025"): ("2025-12-31", "2026-01-01"),
        ("nov 2025", "november 2025", "q3 2025"): ("2025-09-30", "2025-10-01"),
        ("aug 2025", "august 2025", "q2 2025"):   ("2025-06-30", "2025-07-01"),
        ("may 2025", "q1 2025"):                   ("2025-03-31", "2025-04-01"),
    }
    target_period = None
    min_filing = None
    for signals, (period, filing_min) in PERIOD_MAP.items():
        if any(s in q_lower for s in signals):
            target_period = period
            min_filing = filing_min
            break

    if target_period is None:
        _LATEST_SIGNALS = ["latest", "most recent", "current", "newest"]
        if any(s in q_lower for s in _LATEST_SIGNALS):
            periods = [d.metadata.get("period_of_report", "") for d in docs
                       if d.metadata.get("period_of_report", "") not in ("", "UNKNOWN")]
            if periods:
                target_period = max(periods)

    if target_period is None:
        return docs
    filtered = [
        d for d in docs
        if d.metadata.get("period_of_report", "") == target_period
        or (d.metadata.get("period_of_report", "") in ("", "UNKNOWN")
            and min_filing and d.metadata.get("filing_date", "") >= min_filing)
    ]
    return filtered if filtered else docs

# ── Fund diversity filter ─────────────────────────────────────────────────────
def _diversify_by_fund(docs, max_per_fund=2):
    fund_counts = {}
    diverse = []
    for d in docs:
        fund = d.metadata.get("fund_name", "")
        fund_counts[fund] = fund_counts.get(fund, 0) + 1
        if fund_counts[fund] <= max_per_fund:
            diverse.append(d)
    return diverse if diverse else docs

print(f"\nPipeline ready.")
print(f"Filters   : entity + temporal + diversity")
print("Architecture:")
print(f"Retrieval : {RETRIEVAL_LABEL}")
print(f"BM25      : {len(_documents)} docs, k=10")
print(f"Store     : {vectorstore._collection.count()} vectors")
print(f"  Generator  : {GENERATOR_MODEL_NAME} via Google AI Studio (Google DeepMind)")
print(f"  Evaluator  : {EVALUATOR_MODEL} via GitHub Models (OpenAI)")
print(f"  Judge      : {JUDGE_MODEL} via Google AI Studio (Google DeepMind)")
print(f"  → Two different providers (Google / OpenAI) — no self-grading bias.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8584.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Rebuilding BM25 retriever from source JSON ...
FlashrankRerank       : ACTIVE (narrow=8, wide=20)

Pipeline ready.
Filters   : entity + temporal + diversity
Architecture:
Retrieval : Hybrid BM25+Vector + FlashrankRerank
BM25      : 95377 docs, k=10
Store     : 95377 vectors
  Generator  : gemini-3.1-flash-lite-preview via Google AI Studio (Google DeepMind)
  Evaluator  : gpt-4o-mini via GitHub Models (OpenAI)
  Judge      : gemini-2.5-pro via Google AI Studio (Google DeepMind)
  → Two different providers (Google / OpenAI) — no self-grading bias.


In [12]:
# ── Audit Logging Layer ────────────────────────────────────────────────────────
# Every query is fully recorded in system_audit_trail.json.
# Each entry includes the accession_number for SEC regulatory traceability.
# Entries are appended without overwriting the existing log.


def _load_audit_log() -> list:
    """Load existing audit log, or return empty list if file does not exist."""
    if os.path.exists(AUDIT_LOG_PATH):
        with open(AUDIT_LOG_PATH, "r", encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                return []
    return []


def _save_audit_log(entries: list) -> None:
    with open(AUDIT_LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(entries, f, indent=2, ensure_ascii=False)


def audited_rag_query(question: str, query_type: str = "Cross-Fund Sector Synthesis",
                      top_k: int = 8) -> dict:
    """
    Runs the full RAG pipeline (Hybrid Retrieval → Reranker → Stuff/Map-Reduce)
    with dual-judge scoring and grounding check.

    Appends a complete audit record to system_audit_trail.json including
    accession_number for every retrieved chunk.

    Returns the audit entry dict for immediate inspection.
    """
    timestamp = datetime.now(timezone.utc).isoformat()

    # ── 1. Retrieve — use wide reranker for broad queries ─────────────────────
    target_funds = _extract_fund_entities(question)
    is_broad = len(target_funds) == 0
    retriever = reranking_retriever_wide if is_broad else reranking_retriever

    try:
        docs = retriever.invoke(question)
    except Exception as exc:
        print(f"[WARN] Reranker failed ({exc}); falling back to hybrid retriever")
        docs = hybrid_retriever.invoke(question)[:top_k]

    # ── 1b. Entity-scoped filter ──────────────────────────────────────────────
    if target_funds:
        docs = _entity_filter(docs, target_funds)

    # ── 1c. Temporal filter ───────────────────────────────────────────────────
    docs = _temporal_filter(docs, question)

    # ── 1d. Fund diversity filter for broad queries ───────────────────────────
    if is_broad:
        docs = _diversify_by_fund(docs, max_per_fund=2)

    # ── 2. Build the retrieved-context audit records ───────────────────────────
    retrieved_context = []
    for doc in docs:
        m = doc.metadata
        retrieved_context.append({
            "fund_name"        : m.get("fund_name", "unknown"),
            "cik"              : m.get("cik", "unknown"),
            "accession_number" : m.get("accession_number", "unknown"),
            "filing_date"      : m.get("filing_date", "unknown"),
            "period_of_report" : m.get("period_of_report", "unknown"),
            "chunk_index"      : m.get("chunk_index"),
            "total_chunks"     : m.get("total_chunks"),
            "relevance_score"  : round(float(m.get("relevance_score", -1.0)), 6),
            "snippet"          : doc.page_content[:400],
        })

    # ── 3. Route to Stuff or Map-Reduce ───────────────────────────────────────
    chain_label = "Map-Reduce" if query_type in MAP_REDUCE_TYPES else "Stuff"

    if query_type in MAP_REDUCE_TYPES:
        llm_response = run_map_reduce_chain(question, docs)
    else:
        llm_response = run_stuff_chain(question, docs)

    # ── 4. Dual-judge scoring ─────────────────────────────────────────────────
    avg, e_score, e_rat, e_method, j_score, j_rat, j_method = \
        score_dual(llm_response, question, docs)

    # ── 5. Grounding check ────────────────────────────────────────────────────
    grounded, grounding_detail = grounding_check(llm_response, FILINGS_BASE)

    # ── 6. Citations check ────────────────────────────────────────────────────
    response_lower = llm_response.lower()
    fund_names_in_context = [c["fund_name"].lower() for c in retrieved_context]
    cited_funds = [
        name for name in set(fund_names_in_context) if name in response_lower
    ]
    citations_passed = len(cited_funds) > 0

    # ── 7. Collect all unique accession numbers ───────────────────────────────
    accession_numbers = list({
        ctx["accession_number"] for ctx in retrieved_context
        if ctx["accession_number"] != "unknown"
    })

    # ── 8. Build audit entry ──────────────────────────────────────────────────
    entry = {
        "timestamp"              : timestamp,
        "input_query"            : question,
        "query_type"             : query_type,
        "chain_type"             : chain_label,
        "retrieval_method"       : RETRIEVAL_LABEL,
        "generator"              : GENERATOR_MODEL_NAME,
        "accession_numbers"      : accession_numbers,
        "retrieved_context"      : retrieved_context,
        "llm_response"           : llm_response,
        "scoring" : {
            "dual_avg"           : avg,
            "evaluator_score"    : e_score,
            "evaluator_rationale": e_rat,
            "evaluator_method"   : e_method,
            "judge_score"        : j_score,
            "judge_rationale"    : j_rat,
            "judge_method"       : j_method,
        },
        "grounding" : {
            "passed"             : grounded,
            "detail"             : grounding_detail,
        },
        "citations_check" : {
            "passed"             : citations_passed,
            "cited_funds"        : list(set(cited_funds)),
            "context_funds"      : list(set(fund_names_in_context)),
        },
    }

    # ── 9. Append and persist ─────────────────────────────────────────────────
    log = _load_audit_log()
    log.append(entry)
    _save_audit_log(log)

    print(f"[AUDIT] Entry #{len(log)} written to {AUDIT_LOG_PATH}")
    print(f"  Query      : {question[:80]}")
    print(f"  Chain      : {chain_label}  |  Chunks: {len(docs)}")
    print(f"  Score      : {avg}/5 (Eval={e_score}, Judge={j_score})")
    print(f"  Grounded   : {grounded}  — {grounding_detail[:80]}")
    print(f"  Citations  : {'PASSED' if citations_passed else 'FAILED'}")
    print(f"  Accessions : {accession_numbers}")
    return entry


print("Audit logging layer ready.  Call  audited_rag_query(question)  to run.")

Audit logging layer ready.  Call  audited_rag_query(question)  to run.


In [13]:

# ── Run the audited query ──────────────────────────────────────────────────────
AUDITED_QUERY = "Identify the consensus investment trends across all funds for the Feb 2026 filings."

entry = audited_rag_query(AUDITED_QUERY, query_type="Cross-Fund Sector Synthesis", top_k=8)

print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(entry["llm_response"])


[AUDIT] Entry #1 written to ./system_audit_trail.json
  Query      : Identify the consensus investment trends across all funds for the Feb 2026 filin
  Chain      : Map-Reduce  |  Chunks: 4
  Score      : 3.5/5 (Eval=4, Judge=3)
  Grounded   : False  — Verified CUSIPs: ['68389X105', '98978V103', '682680103', '921910840', '921946794
  Citations  : PASSED
  Accessions : ['0000019617-26-000083', '0001761054-26-000002']

FINAL REPORT
Based on the provided 13F filing excerpts for the period ending 2025-12-31, filed in February 2026, here is the analysis of the investment data:

### **Data Availability**
The following funds were requested and are present in the provided summaries:
*   **Treasurer of the State of North Carolina** (Accession Number: 0001761054-26-000002)
*   **JPMORGAN CHASE & CO** (Reported under Bank of America Corp header; Accession Number: 0000019617-26-000083)

### **Consolidated Investment Trends**
The provided data consists exclusively of filings for the period ending *

In [ ]:
# ── Pretty-print the last N audit log entries ─────────────────────────────────
def _load_audit_log_safe() -> list:
    # Use existing pipeline loader if available (cell with audit layer already run)
    existing_loader = globals().get("_load_audit_log")
    if callable(existing_loader):
        return existing_loader()

    # Fallback for out-of-order execution
    os_mod = globals().get("os") or __import__("os")
    json_mod = globals().get("json") or __import__("json")
    audit_path = globals().get("AUDIT_LOG_PATH", "./system_audit_trail.json")

    if os_mod.path.exists(audit_path):
        with open(audit_path, "r", encoding="utf-8") as f:
            try:
                return json_mod.load(f)
            except json_mod.JSONDecodeError:
                return []
    return []


def pretty_print_audit_log(n: int = 3) -> None:
    log = _load_audit_log_safe()
    if not log:
        print("Audit log is empty.")
        return

    recent = log[-n:]
    print(f"\nAUDIT LOG  —  last {len(recent)} of {len(log)} entries")
    print("═" * 70)

    for i, e in enumerate(recent, 1):
        cc = e["citations_check"]
        entry_num = len(log) - len(recent) + i
        print(f"\nEntry {entry_num}  [{e['timestamp']}]")
        print(f"  Retrieval : {e.get('retrieval_method', 'unknown')}")
        print(f"  Query     : {e['input_query'][:100]}")
        print(f"  Chunks    : {len(e['retrieved_context'])} retrieved")
        print("  Sources   :")
        for ctx in e["retrieved_context"][:4]:
            print(f"    - [{ctx['fund_name']}]  "
                  f"CIK: {ctx['cik']}  "
                  f"Accession: {ctx['accession_number']}  "
                  f"score: {ctx.get('relevance_score', '?')}  "
                  f"chunk {ctx['chunk_index']}/{ctx['total_chunks']}")
        if len(e["retrieved_context"]) > 4:
            print(f"    ... and {len(e['retrieved_context']) - 4} more")
        print(f"  Response  : {e['llm_response'][:200].strip()} ...")
        print(f"  Citations check : {'PASSED' if cc['passed'] else 'FAILED'}")
        if cc["cited_funds"]:
            print(f"    Cited funds : {cc['cited_funds']}")
        else:
            print("    No fund names from context found in response.")
        print("  " + "─" * 66)


pretty_print_audit_log(3)


Audit log is empty.
